# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Example for Section 1: Unit of analysis + time window

**Unit of Analysis:** One row represents one `search_query` made by a `user_id` within a specific `session_id`.

**Time Window:** The data covers `YYYY-MM-DD` to `YYYY-MM-DD`.

_Replace this text with your specific unit of analysis and time window._

In [ ]:
# Example: Verify the unit of analysis and time window
# You will need to load your data first. For example, if you're loading from BigQuery:
# import pandas_gbq
# BQ_PROJECT = 'your-gcp-project-id' # Replace with your project ID
# TABLE_ID = 'your-dataset.your-table' # Replace with your table ID
# df = pandas_gbq.read_gbq(f"SELECT * FROM `{TABLE_ID}` LIMIT 1000", project_id=BQ_PROJECT)

# For demonstration, let's create a dummy DataFrame if no data is loaded yet
import pandas as pd
if 'df' not in locals():
    data = {
        'session_id': [1, 1, 2, 2, 3],
        'user_id': ['A', 'A', 'B', 'B', 'C'],
        'search_query': ['apple', 'orange', 'banana', 'apple', 'grape'],
        'query_timestamp': pd.to_datetime(['2023-01-01 10:00:00', '2023-01-01 10:05:00', '2023-01-02 11:00:00', '2023-01-02 11:10:00', '2023-01-03 12:00:00']),
        'clicked_result': [1, 0, 1, 0, 1]
    }
    df = pd.DataFrame(data)

# Verify distinct unit of analysis (example: combination of session, user, search query)
print("Number of unique (session_id, user_id, search_query) combinations:", df[['session_id', 'user_id', 'search_query']].drop_duplicates().shape[0])
print("Total number of rows:", df.shape[0])

# Verify time window
min_date = df['query_timestamp'].min()
max_date = df['query_timestamp'].max()
print(f"Data covers from {min_date} to {max_date}")

# Check for duplicates based on the defined unit of analysis
unit_cols = ['session_id', 'user_id', 'search_query', 'query_timestamp'] # Add other relevant columns for uniqueness
duplicates = df[df.duplicated(subset=unit_cols, keep=False)]
print(f"\nNumber of duplicate rows based on {unit_cols}: {duplicates.shape[0]}")
if not duplicates.empty:
    print("Sample of duplicate rows:")
    display(duplicates.head())

_Replace this code with your actual data loading and verification logic._

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Example for Section 2: Fields: feature / label / context / excluded

Here's a categorization of the fields:

**Features (Inputs for prediction):**
*   `search_query`: The actual text query (string).
*   `query_length`: Length of the search query (integer).
*   `query_language`: Language detected for the query (categorical: 'en', 'es', etc.).
*   `device_type`: Type of device used (categorical: 'mobile', 'desktop', 'tablet').
*   `user_agent_category`: Categorized user agent (categorical).

**Label (What we want to predict):**
*   `clicked_result`: Binary indicator if a result was clicked (0/1).

**Context (Informative, but not direct features/labels):**
*   `session_id`: Unique identifier for a user session (string).
*   `user_id`: Unique identifier for the user (string).
*   `query_timestamp`: Timestamp of the search query (datetime).
*   `search_rank`: Rank of the clicked result (integer).

**Excluded (Why excluded):**
*   `ip_address`: Excluded due to privacy concerns and its high cardinality making it difficult to use directly as a feature.
*   `raw_user_agent`: Excluded because `user_agent_category` provides a more aggregated and usable version, and the raw string is too verbose for direct feature use.

_Replace this categorization with your specific fields and their roles._

In [ ]:
# Example: Inspect DataFrame columns to inform field categorization
print("DataFrame columns and their types:")
display(df.info())

print("\nFirst 5 rows of the DataFrame:")
display(df.head())

print("\nDescriptive statistics for numerical columns:")
display(df.describe())

print("\nNumber of unique values for categorical columns (example):")
for col in ['device_type', 'query_language']:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()} unique values")

_Use this code to get an overview of your data's structure and contents._

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Example for Section 3: Verify it with queries

Each claim from Sections 1 and 2 should be backed by a query. Here are examples:

**1. Grain Verification:** Confirming the uniqueness of the defined unit of analysis.

**2. Counts Verification:** Checking overall row counts, as well as counts for key categorical features or ranges for numerical ones.

**3. Missing Values Check:** Identifying columns with nulls and their percentages.

**4. Time Window Verification:** Ensuring the date range is as expected and there are no unexpected gaps or overlaps if dealing with multiple data sources.

_Detail your specific verification steps and expected outcomes here._

In [ ]:
# Verification Query 1: Grain Verification
# Re-check for duplicates based on the definitive unit of analysis
unit_of_analysis_cols = ['session_id', 'user_id', 'search_query', 'query_timestamp'] # Adjust these columns as per your definition
duplicate_rows = df[df.duplicated(subset=unit_of_analysis_cols, keep=False)]
print(f"Number of rows violating the defined grain (duplicates): {duplicate_rows.shape[0]}")
if not duplicate_rows.empty:
    print("Sample violating rows:")
    display(duplicate_rows.sort_values(by=unit_of_analysis_cols).head())
else:
    print("Grain is unique as defined.")

_Adjust `unit_of_analysis_cols` to match your exact unit._

In [ ]:
# Verification Query 2: Overall Counts and Categorical Distribution
print(f"Total number of records: {df.shape[0]}")

# Example: Count of clicks vs non-clicks (label distribution)
if 'clicked_result' in df.columns:
    print("\nDistribution of 'clicked_result':")
    display(df['clicked_result'].value_counts(normalize=True))

# Example: Distribution of device types
if 'device_type' in df.columns:
    print("\nDistribution of 'device_type':")
    display(df['device_type'].value_counts(normalize=True))

_Add more counts and distributions relevant to your features and labels._

In [ ]:
# Verification Query 3: Missing Values Check
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percent})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing %', ascending=False)

print("\nMissing values in each column:")
if not missing_df.empty:
    display(missing_df)
else:
    print("No missing values found.")

_Review and decide on imputation or dropping strategies based on these findings._

In [ ]:
# Verification Query 4: Time Window Check
if 'query_timestamp' in df.columns:
    print(f"\nMin timestamp: {df['query_timestamp'].min()}")
    print(f"Max timestamp: {df['query_timestamp'].max()}")
    print(f"Total unique days: {df['query_timestamp'].dt.date.nunique()}")

    # Check for gaps if a continuous time window is expected
    # This is a simplified check; a more robust one might involve checking for missing dates in a date range
    expected_days = (df['query_timestamp'].max().date() - df['query_timestamp'].min().date()).days + 1
    actual_days = df['query_timestamp'].dt.date.nunique()
    if actual_days != expected_days:
        print(f"Warning: Expected {expected_days} unique days, but found {actual_days}. There might be gaps in the data.")

_Confirm your time window and check for unexpected anomalies._

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Example for Section 4: Data limits

This section describes what the current data _cannot_ tell you or limitations in its representativeness.

*   **Unbalanced History:** The data might be heavily biased towards certain user segments or query types. For example, if the data only captures logged-in users, it won't reflect anonymous search behavior.
*   **GSC-only early rows:** If data collection methods have changed, older data might only reflect Google Search Console (GSC) data, which has a different granularity or set of metrics compared to newer, richer data sources.
*   **Window Overlaps:** If combining data from multiple sources, there might be implicit overlaps or discrepancies in how time windows are defined, leading to double-counting or skewed aggregations.
*   **Lack of Negative Samples:** For click prediction, if the dataset only contains records where a click occurred, it would be challenging to learn what *doesn't* lead to a click without generating negative samples.
*   **Feature Bias:** Some features might be inherently biased or incomplete (e.g., `query_language` might rely on imperfect detection).
*   **Short Time Horizon:** The available `9 days` of data (from the prompt example) may not be sufficient to capture long-term trends, seasonality, or rare events.

_List the specific limitations of your dataset that could impact model performance or interpretation._

In [ ]:
# Example: Code to illustrate a data limit (e.g., checking data completeness over time)
# This is more illustrative than a strict 'query'

if 'query_timestamp' in df.columns:
    daily_counts = df.set_index('query_timestamp').resample('D').size()
    print("\nDaily record counts:")
    display(daily_counts)

    if (daily_counts == 0).any():
        print("Warning: Some days have zero records, indicating potential data gaps or incomplete history.")
    else:
        print("Data appears to be present for all days within the observed range.")

_Use this space for any code that highlights or verifies the identified data limits._

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.